

![](https://raw.githubusercontent.com/wateraccounting/WaPORMOOC/main/images/banner_notebooks_WaPOR4Global.png)


<div style="text-align:center; margin-top:15px;">
<h3 style= "margin-bottom:5px;">
<i>Module Two</i> – <b>Accessing WaPOR data</b>
</h3>

<p style="margin-top:5px;">
<b>Notebook:</b> Download WaPOR data to local machine &nbsp; | &nbsp;
<b>Estimated time:</b> 2 hours &nbsp; | &nbsp;
<b>Instructor:</b> Dr. Solomon Seyoum
</p>
</div>

[![](https://raw.githubusercontent.com//wateraccounting/WaPORMOOC/main/images/colab-badge.png)](https://colab.research.google.com/github/wateraccounting/WaPOR4Global/blob/main/1.%20Accessing%20WaPOR%20data/Notebook1_Accessing_WaPOR_data.ipynb?target="_blank")




### 🎯 Learning Objectives

When you complete this notebook, you will be able to:

- Explain the different ways how WaPOR data can be accessed  
- Understand the code in the notebook
- Access WaPOR data and download the data needed for module 3  
- Modify the code to access and download the data for your needs    

### 📝 Prerequisite Courses

✔ [Introduction to WaPORv3](https://ocw.un-ihe.org/course/view.php?id=263)  
✔ Geospatial data concepts  
✔ [Python basics for geeospatial data analysis](https://ocw.un-ihe.org/course/view.php?id=272)

### 📖 WaPOR data access

The scripts used in this notebook are built with the WaPOR Version 3 API to download and preprocess different types for data for a selected area(s) of interest and store them in different formats (raster or timeseries):

*   Download raster images for the area of interest and store in a netCDF format
*   Download timeseries from polygons and store in a csv format
*   Download timeseries for points and store in a csv format
*   Download timeseries for a polygon and masked for land cover or crop type and store as csv file

NOTE: The scripts also explain how to download AgERA5 data (available from 1979) useful for long term climatic analyses

After following all the steps in this Notebook you will have downloaded the data required for running the analyses in module 3. See at the end of the Notebook a checklist of the data needed.

---

###  🔀 Workflow Overview
🗺

| Step | Task | Purpose |
|------|------|---------|
| 1 | Environment Setup | importing required packages
| 2 | Input Data Preparation | Set up the input required to run the download script |
| 3 | Running the download script | Downlaad the data |
| 4 | Check| Check if the required data is downloaded based on the specification |
| 5 | Optional - Visualize | Visualize the downloaded data |

---

## ⚙️ 1. Environment Setup
<p style="margin-top:1px;">
	Some Python packages are required that may not be included in a basic Python installation. Run the cells below to set up your environment.
	</p>

In [15]:
# --- Install the required packages
%%capture
!pip install netCDF4
!pip install geopandas
!pip install shapely
!pip install rasterio
!pip install xarray
!pip install dask
!pip install rioxarray
!pip install "dask[distributed]"

In [16]:
# import packages
import os
import shutil
import zipfile
from glob import glob
import xarray as xr
import pandas as pd

In this Notebook we will be using a package called **download_WaPORv3_data** to access the WaPOR API and download WaPORv3 data.

[Download](https://github.com/wateraccounting/WaPORMOOC/blob/main/package/download_WaPOR_v3_data.py) the python code from github, and upload to this session and run the next cell to import the code. Ensure that the directory path where the file is located is the correct one.

In [ ]:
#To upload file.
uploaded = files.upload()

In [17]:
#--- Define directory path where download_WaPOR_v3_data.py is stored
dir_code = '/content'

# --- import required packages
import time
import sys
import importlib
sys.path.append(dir_code) #add folder with local modules to system paths
import download_WaPOR_v3_data as wdl
wld = importlib.reload(wdl)


## ⚙️ 2. Input data preparation
<p style="margin-top:1px;">
It is also good practice to use an organized folder structure. In the next cell we will be creating a folder where the outputs are stored, for each download script you define a folder where the data is stored, this folder is then created in this output folder<br><br>


In [ ]:
dir_output_data = '/content/output_data'
if not os.path.exists(dir_output_data):
    os.makedirs(dir_output_data)
dir_input_data, dir_output_data

For the upcoming exercises you will be using different data sets to explain the different methods of downloading WaPOR data:

*   Wad_Helal.geojson
*   WH_Fields.geojson
*   met_stations.geojson
*   ESA_LC_2021_Erbil_20m.tif

For the exercises in Module 3 (Analysing and visualising WaPOR data) you need the following data:

*   Iraq_admin1.geojson (containing the shapefile for the 18 governorates of Iraq)
*   ESA_LC_2021_Erbil_20m.tif (land cover map obtained from ESA for the Erbil governorate and resampled to 20m resolution)
*   SelectedCPs.geojson (Locations of selected farms in Erbil governorate)


Download these files from the [data folder](https://github.com/wateraccounting/WaPOR4Global/tree/main/data) in the WaPOR4Global repository on github. 

Upload the files to this session. 

---

## 3. Downloading WaPORv3 data

### Information needed to run the download script
The download script has a similar structure as what you are familiar with (as explained in the [Introduction WaPORv3 MOOC](https://ocw.un-ihe.org/course/view.php?id=263)). It requires the following information from the user:
<ul style="margin-top:0px;">
  <li>The folder path where the data should be saved</li>
  <li>The area of interest as a geojson file </li>
  <li>A list of the WaPOR data products needed, including level, variable name, and frequency*</li>
  <li>The start and end dates for the requested data</li>
  <li>There are four options, for each and example is provided in the next section
  
      a) raster data (saved in NetCDF format)
      b) point data for one or multiple locations (saved in csv format)
      c) Zonal statistics using polygons (saved in csv format)
      d) Zonal statistics by raster: for example statistics per land cover class (saved in csv format)
  
  </li>
    
</ul>
*NOTE: It is also possible to download AgERA5 data using this package as will be explained in the exercises.

---

### 3a - Download raster data of an are of interest (AOI)
This notebook, similar to the one in the Introduction to WaPORv3 MOOC downloads raster data from WaPOR. The main difference is that it compiles the data into a netCDF file. If you want individual TIFF files please use the [WaPOR_download script](https://github.com/wateraccounting/WaPORMOOC/blob/main/1_WaPOR_download_colab/Download_WaPORv3_Data.ipynb) from the previous MOOC. We will be downloading the raster files for the Wad Helal irrigation block located in the Gezira irrigation scheme, Sudan (same data as used in the Python for Geospatial analyses using WaPOR data MOOC). 

Steps:
*   Upload a shapefile or geojson to the input data folder
*   Create a folder by putting the name in the first line
*   Update the relative path in the second line
*   Select the products and period of the data you want to download
*   Run the cell
*   Check the folder for the downloaded file

NOTE: data_type is set as **'raster'**

The geojson file of the Wad Helal area can be found in the [data folder in the WaPOR4Global repository](https://github.com/wateraccounting/WaPOR4global/tree/main/data) (*Wad_Helal.geojson*).

In [ ]:
project_foldr = f"{dir_output_data}/Gezira"
region = r"/content/Wad_Helal.geojson"
products = [ "L3-AETI-D", "L3-NPP-D"]
period = ["2022-10-01", "2023-04-30"]
data_type = "raster"

#  --- Run the download script
# %cd /content
tt = time.time()
wdl.wapor_dl(region, products, period, project_foldr, data_type=data_type)

elapsed = time.time() - tt
print(
	">> Time elapsed up to downloading the required data : "
	+ "{0:.1f}".format(elapsed)
	+ " s"
)

If the above cell runs successfully, then you will see the netCDF files created in the folder "output_data/Gezira/nc" one for each variable.

---

### 3b - Download timeseries of point locations

This notebook allows for downloading timeseries for point locations. Upload a shapefile with point locations and select the product and period to download. Notice that the data_type is changed to **'point'**.

Note: To label the columns in the file where the data will be saved, assign the name of the column to points_col_name. Otherwise the index will be used.

We will download daily data for precipitation (PCP) and reference ET (RET) for a number of meteorological stations accross Africa (same locations used in WaPOR concepts and validation MOOC). The geojson file with the met station locations is located in the data folder of the WaPOR4Global repository (*met_stations.geojson*).

In [ ]:

project_foldr = f"{dir_output_data}/Africa"
region = r"/content/met_stations.geojson"
products = ["L1-PCP-E","L1-RET-E"]
period = ["2023-01-01", "2023-12-31"]
data_type = "point"

tt = time.time()
points_col_name = "Station_code"  ## The name of the points id
wdl.wapor_dl(
				region,
				products,
				period,
				project_foldr,
				data_type=data_type,
				points_col_name=points_col_name,
		)

elapsed = time.time() - tt
print(
	">> Time elapsed up to downloading the required data : "
	+ "{0:.1f}".format(elapsed)
	+ " s"
)

If the above cell runs successfully, then you will see CSV files created in the folder "output_data/Africa/csvs" one for each variable.

---

### 3c - Download timeseries for polygons using zonal statistics.

To download timeseries for polygons, the following notebook can be used. Note that the data type is now set at zonal_stat, which requires that the type of statistic needs to be provided, the default is **'Mean'**.

If the shapefile of the AIO has a name column to identify the polygons, it will  be used to lable the columns in the file where the data will be saved. Otherwise the index will be used.

In this Notebook, we will download the WaPOR AETI and NPP data for the farmer fields in the Wad Helal area (use *WH_Fields.geojson*, case study of the MOOC on python for geospatial analyses).

In [ ]:
project_foldr = f"{dir_output_data}/Gezira"
region = r"/content/WH_Fields.geojson"
products = ["L1-AETI-D","L3-NPP-D"]
period = ["2022-10-01", "2023-04-30"]
data_type = "zonal_stat"

tt = time.time()
polygons_col_name = "name"
stat = "mean"  # possible stats include max, min, median
wdl.wapor_dl(
    region,
    products,
    period,
    project_foldr,
    data_type=data_type,
    stat=stat,
    polygons_col_name=polygons_col_name,
)

elapsed = time.time() - tt
print(
	">> Time elapsed up to downloading the required data : "
	+ "{0:.1f}".format(elapsed)
	+ " s"
)

If the above cell runs successfully, then you will see CSV files created in the folder "output_data/Gezira/csvs" one for each variable. See how easy this was?

---

## 📊 **Exercise 1: Download data for Module 3 - part 1**:
It is also possible to download climatic data (PCP and RET) from AgERA5 using the same script. The advantage is that this data is available since 1979.

Change the line products by replacing it with the reference to AgERA5 data as illustrated below:

```
products = ["AgERA5-PCP-M", "AgERA5-RET-M"]
```

For module 3, you have to download **monthly** AgERA5 Precipitation and Reference ET data for the 18 different governorates in Iraq for the period 1980-2025. The geojson file of the Iraqi governorates can be found in the [data folder in the WaPOR4Global repository](https://github.com/wateraccounting/WaPOR4global/tree/main/data) (*irq_admin1.geojson*). Adapt the script above in such a way that it downloads the required data. Save this information in a folder called Iraq.

Ensure that you have the following data
* monthly data for PCP and RET from AgERA5 using the irq_admin1.geojson for the period 1980-2025, stored in folder called Iraq

---

## 📊 **Exercise 2: Download data for Module 3 - part 3:**

For the next module (part 3) you need timeseries of dekadal AETI, NPP and RSM, for selected fields (*SelectedCPs.geojson*) in the Erbil area for the period 2018-2025. Save this information in a folder called CentrePivots. Adapt the script above in such a way that it downloads the required data.

Ensure that you have the following data (needed for part 3 of module 3)
* dekadal data for AETI and NPP using the Erbil_fields.geojson for the period 2018-2025, stored in folder called CentrePivots


## 📊 **Exercise 3: Download additional climate data for Module 3 - part 2&3:**

Additional climate data is needed for the analyses in Module 3, which you can also download using the script provided above:
* monthly PCP and RET data for the Erbil area stored in folder called Erbil (needed for part 2)
* dekadal PCP and RET data for the Erbil area stored in folder called CentrePivots (needed for part 3)

### 3d - Download data - Zonal statistics by raster: for example, statistics per land cover class.

N.B. The path to the raster to define the zones should be provided. In this example we use the land cover raster for the Erbil area in Iraq (ESA_LC_2021_Erbil_20m.tif). The file can be found in the data folder of the WaPOR4Global repository.

The raster file which defines the zones (for example the land cover raster) may have a different resolution than the data to be downloaded. In this case we can chose either to resample the data to be downloaded to match the LCC raster or resample the LCC to match the resolution of the data to be downloaded.

In [ ]:
project_foldr = f"{dir_output_data}/Erbil"
#region = r"/content/Shammamuk.geojson"
products = [ "L2-AETI-M", "L2-NPP-M"]
period = ["2018-01-01", "2025-12-31"]
data_type = "sample_by_lcc"

tt = time.time()
stat = "mean"  # possible stats include max,
lcc_raster_path = r"/content/ESA_LC_2021_Erbil_20m.tif"
wdl.wapor_dl(
    region,
    products,
    period,
    project_foldr,
    data_type=data_type,
    stat=stat,
    lcc_raster_path=lcc_raster_path,
    use_lcc_resolution="no",  # if yes, the data will be resampled to mach the lcc raster
)

elapsed = time.time() - tt
print(
	">> Time elapsed up to downloading the required data : "
	+ "{0:.1f}".format(elapsed)
	+ " s"
)

If the above cell runs successfully, then you will see CSV files created in the folder "output_data/Erbil/csvs" one for each variable (AETI and NPP).

---

## 📊 **Exercise 4: Download data for Module 3 - part 2**

The data you just downloaded is needed for the WaPOR analyses. 

Ensure that you have the following data (needed for part 2 of module 3)
* monthly data for AETI and NPP for land cover cropland for the period 2018-2025, stored in folder called Erbil
---

## 4. Download the data folder to local machine

Ensure you have downloaded the following data needed for the different analyses:

Part 1 long time series analyses
- monthly AgERA5 PCP and RET data for 18 Iraqi governorates from 1980-2025

Part 2 comparative analyses
- monthly L2 NPP and AETI data for cropland in Erbil governorate from 2018-2025
- monthly L1 PCP and RET data Erbil governorate from 2018-2025

Part 3 forecasting
- dekadal L3 NPP, AETI and RSM data for selected fields from 2018-2025
- dekadal L1 PCP and RET data for Erbil governorate from 2018-2025


In [ ]:
source_folder = dir_output_data
destination_folder = "/content"
zip_name = f"Downloaded.zip"

# Run zip command
!zip -r "{destination_folder}/{zip_name}" "{source_folder}"

from google.colab import files
files.download(f"{destination_folder}/{zip_name}")